In [13]:
# random_forest.py - setting up a RF model to predict ET
# Author: Archie Benn
# Date: 12-07-2026

# import libraries
import numpy as np
import pandas as pd
import optuna
import os
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score


In [14]:
# set seed for NumPy
np.random.seed(42)

# setup wd 
os.chdir('/home/ab/Dropbox/university/github/bbinf_project')
os.getcwd()


'/home/ab/Dropbox/university/github/bbinf_project'

In [15]:
# import data
df_14 = pd.read_csv("data/main/14_pre_processing/df_ml_ready.csv")
df_14.head()


,Unnamed: 0,Site_ID,Date,Latitude,Longitude,Site_age,Age_range,ET,LE,Lai_500m,...,Continent_Europe,Continent_North America,Cover_type_DBF,Cover_type_EBF,Cover_type_ENF,Cover_type_MF,Cover_type_OSH,Climate_zone_Dry,Climate_zone_Temperate,Climate_zone_Tropical
0,0,BE-Bra,2005-01-15,51.30761,4.51984,81,51-100,0.132589,3.83379,0.397561,...,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
1,1,BE-Bra,2005-01-16,51.30761,4.51984,81,51-100,0.052709,1.52237,0.386585,...,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
2,2,BE-Bra,2005-01-17,51.30761,4.51984,81,51-100,0.064565,1.85631,0.375610,...,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
3,3,BE-Bra,2005-01-18,51.30761,4.51984,81,51-100,0.105265,3.03227,0.364634,...,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
4,4,BE-Bra,2005-01-19,51.30761,4.51984,81,51-100,0.149545,4.30787,0.353659,...,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0


In [ ]:
# set features and target
y = df_14['ET']

non_features_list = [
                     'Site_ID',       
                     'Latitude',
                     'Longitude',
                     'Age_range',
                     'ET',
                     'LE',
                     'P',              # have cumulative sum already
                     'Pa',             # dropped
                     'Date',
                     'Unnamed: 0',      # not sure where this came from
                     'Site_age'         # leave out or keep in
                     ]


# drop the non features to get a features df
X = df_14.drop(columns=non_features_list)

len(y) == len(X)


True

In [35]:
y.head()


0    0.132589
1    0.052709
2    0.064565
3    0.105265
4    0.149545
Name: ET, dtype: float64

## Test RF model on one site


In [36]:
# set cross validation method to leave one group out
cv = LeaveOneGroupOut()

# set sites as groups to split by  
sites = df_14["Site_ID"] 


In [47]:
# optuna hyperparameter tuning on one site as a test before doing on all
# will allow me to reduce the search area for the larger loop later
def param_searcher(trial):

    # define search zone
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 450, step=100),
        "max_depth": trial.suggest_int("max_depth", 5, 50),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 20),
        "max_features": trial.suggest_float("max_features", 0.3, 1.0),
    }

    # set cross validation method to leave one group out
    cv = LeaveOneGroupOut()

    # set trial test sit
    test_site = "FI-Hyy"

    # run the leaveOneOut generator (cv) until the test index name == test site
    for train_idx, test_idx in cv.split(X, y, groups=sites):

        # if site name in test split == set test site
        if sites.iloc[test_idx].unique()[0] == test_site:
            break

    # set train/test split for features (X df) using indices from the leaveOneOute generator
    # train_idx is the row indices of the training rows (ie. non test site rows, opposite for test_idx)
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]

    # and same for train/test values of target/ET
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    # now actually setup the random forest
    # model params setup
    rf = RandomForestRegressor(

        # use parameters defined by optuna search (** expands dict)
        **params,
        random_state=42,
        # use all except one core
        n_jobs=-2)
    
    # and actually training the model
    rf.fit(X_train, y_train)

    # predictions
    preds = rf.predict(X_test)

    # calculate rmse and return
    # rmse = np.sqrt(mean_squared_error(y_test, preds))
    r2 = r2_score(y_test, preds)
    return r2   


In [48]:
# setup optuna to maximise r2 during the runs
study = optuna.create_study(direction="maximize")

# no run optuna using the param searcher defined above set to maximise r2
study.optimize(param_searcher, n_trials=10)


[I 2026-07-14 17:36:14,468] A new study created in memory with name: no-name-5b4f69ed-c126-4cd4-b285-b1907de87bf6
[I 2026-07-14 17:36:16,907] Trial 0 finished with value: 0.7458126503309215 and parameters: {'n_estimators': 50, 'max_depth': 26, 'min_samples_leaf': 12, 'max_features': 0.34937579179431033}. Best is trial 0 with value: 0.7458126503309215.
[I 2026-07-14 17:36:22,287] Trial 1 finished with value: 0.7479144694570353 and parameters: {'n_estimators': 50, 'max_depth': 48, 'min_samples_leaf': 7, 'max_features': 0.6900979205693718}. Best is trial 1 with value: 0.7479144694570353.
[I 2026-07-14 17:37:27,743] Trial 2 finished with value: 0.7405032485326797 and parameters: {'n_estimators': 450, 'max_depth': 36, 'min_samples_leaf': 4, 'max_features': 0.6138290121036487}. Best is trial 1 with value: 0.7479144694570353.
[I 2026-07-14 17:37:37,407] Trial 3 finished with value: 0.7223257878367189 and parameters: {'n_estimators': 150, 'max_depth': 20, 'min_samples_leaf': 8, 'max_features':

## Optuna results
- ran on a few test sites to get a feel on the parameter setup which works well for R-squared values  
- outputs of best params below from different site runs

```
[I 2026-07-14 14:48:26,628] Trial 5 finished with value: 0.56352086087377 and parameters: {'n_estimators': 350, 'max_depth': 42, 'min_samples_leaf': 5, 'max_features': 0.36298784370629666}. Best is trial 5 with value: 0.5635208615087377.

[I 2026-07-14 15:03:08,223] Trial 4 finished with value: 0.8650570393561807 and parameters: {'n_estimators': 250, 'max_depth': 23, 'min_samples_leaf': 13, 'max_features': 0.37541226649891885}. Best is trial 4 with value: 0.8650570393561807.

[I 2026-07-14 16:31:56,621] Trial 8 finished with value: 0.5725928903159345 and parameters: {'n_estimators': 350, 'max_depth': 17, 'min_samples_leaf': 16, 'max_features': 0.5303062881263381}. Best is trial 8 with value: 0.5725928903159345.

[I 2026-07-14 16:37:10,176] Trial 4 finished with value: 0.682175463245644 and parameters: {'n_estimators': 350, 'max_depth': 7, 'min_samples_leaf': 19, 'max_features': 0.7485711361996105}. Best is trial 4 with value: 0.682175463245644.

[I 2026-07-14 17:39:10,615] Trial 7 finished with value: 0.8298453443610134 and parameters: {'n_estimators': 50, 'max_depth': 8, 'min_samples_leaf': 15, 'max_features': 0.6020589929197737}. Best is trial 7 with value: 0.8298453443610134.


```




In [27]:
# making predictions
preds = rf.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, preds))
r2 = r2_score(y_test, preds)

print(f"Site: {test_site}, RMSE: {rmse:.4f}, R-squared: {r2:.4f}")


Site: US-Me6, RMSE: 0.5535, R-squared: 0.5460
